In [1]:

import sys
import os
sys.path.append('../TCT/')
from TCT import node_normalizer
from TCT import name_resolver
from TCT import translator_metakg
from TCT import translator_kpinfo
from TCT import translator_query
from TCT import TCT_neighborhood_finder

from TCT import TCT


In [4]:
APInames, metaKG, Translator_KP_info= translator_metakg.load_translator_resources(use_new_metakg_url=True)

All_predicates = list(set(metaKG['Predicate']))
All_categories = list((set(list(set(metaKG['Subject']))+list(set(metaKG['Object'])))))
API_withMetaKG = list(set(metaKG['API']))

    # generate a dictionary of API and its predicates
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
Skipping server without x-maturity: {'url': '/sipr'}


In [ ]:
#metaKG.to_csv('../metaData/metaKG.csv', index=False)

In [7]:
# This is an example of selecting a list of APIs for the neighborhood finder. The user can modify this list to include the APIs they want to use. The APIs in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph. The user can also modify the list of predicates to use for finding the neighborhood. The predicates in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph. 
# The user can also modify the list of categories to use for finding the neighborhood. 
# The categories in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph.
# if selected_APIlist is empty, use all APIs in APInames
selected_APIlist = ['Retriever',
                    'Clinical Trials KP - TRAPI 1.5.0',
                    'Drug Approvals KP - TRAPI 1.5.0',
                    'Genetics Data Provider for NCATS Biomedical Translator Reasoners',
                    'Microbiome KP - TRAPI 1.5.0',
                    'MolePro',
                    'COHD TRAPI',
                    'RTX KG2 - TRAPI 1.5.0',
                    'Text Mined Cooccurrence API',
                    'CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0',
                    'CATRAX BigGIM GeneExpression Performance Phase KP - TRAPI 1.5.0',
                    ]

# add Automat API to the selected API list if it is not already in the list
for api in APInames:
    if 'Automat' in api and api not in selected_APIlist:
        selected_APIlist.append(api)
        
# select a list of APIs to use and a list of predicates to use
if len(selected_APIlist) == 0:
    select_APIs = APInames
else:
    select_APIs = {k: APInames[k] for k in selected_APIlist if k in APInames}


selected_metaKG = metaKG[metaKG['API'].isin(select_APIs.keys())]
#print(select_APIs)


All_predicates = list(set(selected_metaKG['Predicate']))
All_categories = list((set(list(set(selected_metaKG['Subject']))+list(set(selected_metaKG['Object'])))))
API_withMetaKG = list(set(selected_metaKG['API']))
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(selected_metaKG[selected_metaKG['API'] == api]['Predicate']))

In [14]:
name_resolver.lookup('acute myeloid leukemia', return_top_response = False)


[TranslatorNode(curie='MONDO:0018874', label='acute myeloid leukemia', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='MONDO:0005223', label='acute myeloid leukemia with minimal differentiation', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='MONDO:0017893', label='inherited acute myeloid leukemia', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='MONDO:0020317', label='acute myeloid leukemia with 11q23 ab

In [16]:
# I would like to find what genes or proteins are associated with myelodysplastic syndrome. I will use the neighborhood finder to find the genes or proteins that are associated with myelodysplastic syndrome. I will use the selected APIs and predicates to find the neighborhood of myelodysplastic syndrome in the knowledge graph. I will then filter the results to only include genes or proteins that are associated with myelodysplastic syndrome.

input_identifiers = 'MONDO:0018874' # acute myeloid leukemia
# to exclude BioThings Explorer (BTE) TRAPI: 
input_node_id, result, result_parsed, result_ranked_by_primary_infores = TCT_neighborhood_finder.neighborhood_finder(input_identifiers,
                                                                                            node2_categories = ['biolink:Gene','biolink:Protein'],
                                                                                            # node2_categories = ['biolink:Disease','biolink:Protein', 'biolink:Gene'],
                                                                                            APInames = select_APIs,
                                                                                            metaKG = selected_metaKG,
                                                                                            API_predicates = API_predicates)     

TCT_neighborhood_finder_result = TCT_neighborhood_finder.parse_results_for_neighborhood_finder(input_identifiers, result,
        start_node_categories='biolink:Disease', end_node_categories=None,
        get_node_info=True,
        scoring_method='infores')

# write a result to a json file
import json
# add a timestamp to the file name
import datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
with open('TCT_neighborhood_finder_result_'+input_identifiers.replace(':', '_')+'_'+timestamp+'.json', 'w') as f:
    json.dump(TCT_neighborhood_finder_result, f)

# please find the result in your current working directory

MONDO:0018874
Automat-monarchinitiative(Trapi v1.5.0): Success!
Genetics Data Provider for NCATS Biomedical Translator Reasoners: Success!
Automat-pharos(Trapi v1.5.0): Success!
Automat-robokop(Trapi v1.5.0): Success!
CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0: Success!
Retriever: Success!
Clinical Trials KP - TRAPI 1.5.0: Success!
RTX KG2 - TRAPI 1.5.0: Success!


In [19]:

input_identifiers = 'NCBIGene:4869'
input_node_id, result, result_parsed, result_ranked_by_primary_infores = TCT_neighborhood_finder.neighborhood_finder(input_identifiers,
                                                                                            node2_categories = ['biolink:Drug','biolink:SmallMolecule'],
                                                                                            # node2_categories = ['biolink:Disease','biolink:Protein', 'biolink:Gene'],
                                                                                            APInames = select_APIs,
                                                                                            metaKG = selected_metaKG,
                                                                                            API_predicates = API_predicates)     

TCT_neighborhood_finder_result = TCT_neighborhood_finder.parse_results_for_neighborhood_finder(input_identifiers, result,
        start_node_categories='biolink:Gene', end_node_categories=None,
        get_node_info=True,
        scoring_method='infores')

# write a result to a json file
import json
# add a timestamp to the file name
import datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
with open('TCT_neighborhood_finder_result_'+input_identifiers.replace(':', '_')+'_'+timestamp+'.json', 'w') as f:
    json.dump(TCT_neighborhood_finder_result, f)

NCBIGene:4869
RTX KG2 - TRAPI 1.5.0: Success!
Automat-hetionet(Trapi v1.5.0): Success!
Automat-cam-kp(Trapi v1.5.0): Success!
Automat-ctd(Trapi v1.5.0): Success!
MolePro: Success!
Automat-robokop(Trapi v1.5.0): Success!
NodeNorm does not know about these identifiers: ttd.target:IPP-204106,CHEBI:110200,CHEBI:14222,CHEBI:232810
